# 00 — Project overview and data strategy

This notebook recorded the data-access decision and the modeling plan before any model was trained. The project title referred to mobile-money, USSD, and airtime-top-up data. Those records were sensitive individual-level financial and telecommunications traces, so the project avoided pretending that such data had been bundled.

The runnable supervised workflow used the UCI Default of Credit Card Clients dataset. The feature-engineering layer then translated repayment histories, bill amounts, and payment behavior into alternative-data-inspired features. This kept the work reproducible while preserving an honest boundary between public proxy data and private mobile-money data.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from project_package.config import ensure_project_dirs, PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, TARGET
from project_package.plotting import save_figure
from project_package.reporting import save_table

ensure_project_dirs()
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

## Modeling framing

The project was framed as a binary classification problem. The target represented default in the next month. The model output was interpreted as a probability of default, because credit scoring required ranked risk estimates and threshold decisions rather than only hard labels.

In [2]:
project_plan = pd.DataFrame([
    {"stage": "data", "decision": "Downloading the UCI public credit-default dataset during Notebook 01", "reason": "It provided a legally accessible supervised default target."},
    {"stage": "features", "decision": "Creating repayment and transaction-style proxy features", "reason": "These features approximated the behavioral summaries used in alternative-data lending."},
    {"stage": "models", "decision": "Comparing logistic regression, random forest, XGBoost, and a neural network", "reason": "The comparison separated interpretable baselines from high-capacity learners."},
    {"stage": "evaluation", "decision": "Using ROC-AUC, PR-AUC, Brier score, log loss, calibration, lift, and threshold-cost curves", "reason": "Credit scoring required discrimination, calibration, and business-cost assessment."},
    {"stage": "governance", "decision": "Adding basic group-error diagnostics and adverse-action reason helpers", "reason": "Alternative-data credit scoring could create privacy and fairness risks."},
])
save_table(project_plan, "00_project_plan.csv")
project_plan

,stage,decision,reason
0,data,Downloading the UCI public credit-default data...,It provided a legally accessible supervised de...
1,features,Creating repayment and transaction-style proxy...,These features approximated the behavioral sum...
2,models,"Comparing logistic regression, random forest, ...",The comparison separated interpretable baselin...
3,evaluation,"Using ROC-AUC, PR-AUC, Brier score, log loss, ...","Credit scoring required discrimination, calibr..."
4,governance,Adding basic group-error diagnostics and adver...,Alternative-data credit scoring could create p...


## Data sources and substitutions

The table below documented which datasets were used, which were optional, and which were not included.

In [3]:
data_position = pd.DataFrame([
    {"source": "UCI Default of Credit Card Clients", "role": "primary runnable supervised dataset", "included": "downloaded by notebook", "risk": "not mobile-money data"},
    {"source": "Mobile-money / USSD / airtime provider records", "role": "target real-world data type", "included": "not included", "risk": "sensitive private data; normally not open"},
    {"source": "Mobile-money feature dictionary", "role": "schema template", "included": "included as metadata", "risk": "not real customer records"},
    {"source": "Home Credit Default Risk", "role": "optional extension", "included": "not bundled", "risk": "requires user download under Kaggle terms"},
    {"source": "World Bank Global Findex", "role": "financial-inclusion context", "included": "referenced", "risk": "country-level/individual survey context, not default labels"},
])
save_table(data_position, "00_data_access_position.csv")
data_position

,source,role,included,risk
0,UCI Default of Credit Card Clients,primary runnable supervised dataset,downloaded by notebook,not mobile-money data
1,Mobile-money / USSD / airtime provider records,target real-world data type,not included,sensitive private data; normally not open
2,Mobile-money feature dictionary,schema template,included as metadata,not real customer records
3,Home Credit Default Risk,optional extension,not bundled,requires user download under Kaggle terms
4,World Bank Global Findex,financial-inclusion context,referenced,"country-level/individual survey context, not d..."


## Expected final report

After running all notebooks, the generated tables and figures could support a report on credit-default prediction using open repayment data and alternative-data-inspired feature abstraction. The report should not claim that real mobile-money data had been analyzed unless a legitimate provider dataset was later added.